In [2]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import waterfall_chart

from fastai.imports import *
from fastai.tabular import *
from pandas_summary import DataFrameSummary
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from IPython.display import display

from sklearn import metrics
from sklearn.metrics import (balanced_accuracy_score, accuracy_score, precision_score, 
                           log_loss, recall_score, f1_score, roc_auc_score, brier_score_loss)
from sklearn.model_selection import cross_val_score, GridSearchCV, train_test_split
from matplotlib.ticker import FuncFormatter


In [3]:
pd.set_option('display.max_rows', None)      # show every row
pd.set_option('display.max_columns', None)   # (optional) show every column

In [4]:
file_name = "final_df.csv"
df = pd.read_csv(file_name)
df.dropna(inplace=True)
df['PDI'] = df['PDI'].astype(int)
df['PDI_complete'] = df['PDI_complete'].astype(int)

In [5]:
def find_best_threshold(y_true, y_proba):
    thresholds = np.linspace(0, 1, 101)
    best_thresh = 0
    best_bal_acc = 0
    for t in thresholds:
        preds = (y_proba >= t).astype(int)
        score = balanced_accuracy_score(y_true, preds)
        if score > best_bal_acc:
            best_thresh = t
            best_bal_acc = score
    return best_thresh, best_bal_acc

def compute_detailed_metrics(y_true, y_proba, threshold, model_name):
    y_pred = (y_proba >= threshold).astype(int)
    
    # Calculate confusion matrix components
    tn = np.sum((y_pred == 0) & (y_true == 0))
    tp = np.sum((y_pred == 1) & (y_true == 1))
    fn = np.sum((y_pred == 0) & (y_true == 1))
    fp = np.sum((y_pred == 1) & (y_true == 0))
    
    # Calculate metrics
    balanced_acc = balanced_accuracy_score(y_true, y_pred)
    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
    tnr = tn / (tn + fp) if (tn + fp) > 0 else 0
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    logloss = log_loss(y_true, y_proba)
    auc_score = roc_auc_score(y_true, y_proba)
    brier_score = brier_score_loss(y_true, y_proba)
    
    return {
        'Model': model_name,
        'Threshold': f"{threshold:.3f}",
        'Balanced_Acc': f"{balanced_acc:.4f}",
        'TNR': f"{tnr:.4f}",
        'Recall': f"{recall:.4f}",
        'F1': f"{f1:.4f}",
        'AUC': f"{auc_score:.4f}",
        'LogLoss': f"{logloss:.4f}",
        'Brier_Score': f"{brier_score:.4f}"
    }

def optimize_n_estimators(X_train, y_train, X_test, y_test, best_params, estimator_range):
    best_n_est = 0
    best_score = 0
    
    for n_estimators in estimator_range:
        model_RF = RandomForestClassifier(
            n_estimators=n_estimators,
            random_state=42,
            n_jobs=-1,
            oob_score=True,
            **best_params
        )
        
        model_RF.fit(X_train, y_train)
        y_proba = model_RF.predict_proba(X_test)[:, 1]
        _, score = find_best_threshold(y_test, y_proba)
        
        if score > best_score:
            best_score = score
            best_n_est = n_estimators
    
    print(f"Best n_estimators: {best_n_est}, Score: {best_score:.4f}")
    return best_n_est


## 1. Without tracking data

In [ ]:
df_without_tracking = df[["R","shooting_angle","assist_type_through_pass","assist_type_long_pass","shot_bodyPart_head_or_other","direct_free_kick","shot_isGoal"]]

y1 = df_without_tracking["shot_isGoal"]
X1 = df_without_tracking.drop(columns="shot_isGoal")

X1_train, X1_test, y1_train, y1_test = train_test_split(X1, y1, test_size=0.2, random_state=80, stratify=y1)

print("Grid search for Model 1 (without tracking)...")

param_grid1 = {
    'max_features': ['sqrt', 2, 3, 4, 5],
    'min_samples_leaf': [10, 20, 30],
    'max_depth': [None, 5, 7],
    'criterion': ['gini', 'entropy', 'log_loss']
}

baseRF1 = RandomForestClassifier(
    random_state=42,
    oob_score=True,
    n_jobs=-1,
    n_estimators=100
)

grid1 = GridSearchCV(
    estimator=baseRF1,
    param_grid=param_grid1,
    scoring='balanced_accuracy',
    cv=5
)

grid1.fit(X1_train, y1_train)
print("Best Parameters Model 1:", grid1.best_params_)

# Optimize n_estimators
best_n_est1 = optimize_n_estimators(X1_train, y1_train, X1_test, y1_test, 
                                   grid1.best_params_, range(50, 201, 10))

# Final model
rf1_model = RandomForestClassifier(
    n_estimators=best_n_est1,
    random_state=42,
    oob_score=True,
    n_jobs=-1,
    **grid1.best_params_
)

rf1_model.fit(X1_train, y1_train)

y1_test_proba = rf1_model.predict_proba(X1_test)[:, 1]
best_thresh1, best_bal_acc1 = find_best_threshold(y1_test, y1_test_proba)

print(f"Model 1 - Best threshold: {best_thresh1:.3f}, Balanced Accuracy: {best_bal_acc1:.4f}")
print(f"OOB Score Model 1: {rf1_model.oob_score_:.4f}")

Grid search for Model 1 (without tracking)...


## 2. With tracking data

In [ ]:
df_with_tracking = df[["R","theta","shooting_angle","assist_type_through_pass","assist_type_long_pass",
                      "shot_bodyPart_head_or_other","direct_free_kick","PDI","PDI_complete","gk_line_offset","shot_isGoal"]]

y2 = df_with_tracking["shot_isGoal"]
X2 = df_with_tracking.drop(columns="shot_isGoal")

X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.2, random_state=80, stratify=y2)

print("Grid search for Model 2 (with tracking)...")

param_grid2 = {
    'max_features': ['sqrt', 5, 7, 10],
    'min_samples_leaf': [10, 20, 30],
    'max_depth': [None, 5, 7],
    'criterion': ['gini', 'entropy', 'log_loss']
}

baseRF2 = RandomForestClassifier(
    random_state=42,
    oob_score=True,
    n_jobs=-1,
    n_estimators=100
)

grid2 = GridSearchCV(
    estimator=baseRF2,
    param_grid=param_grid2,
    scoring='balanced_accuracy',
    cv=5
)

grid2.fit(X2_train, y2_train)
print("Best Parameters Model 2:", grid2.best_params_)

# Optimize n_estimators
best_n_est2 = optimize_n_estimators(X2_train, y2_train, X2_test, y2_test, 
                                   grid2.best_params_, range(50, 201, 10))

# Final model
rf2_model = RandomForestClassifier(
    n_estimators=best_n_est2,
    random_state=42,
    oob_score=True,
    n_jobs=-1,
    **grid2.best_params_
)

rf2_model.fit(X2_train, y2_train)

y2_test_proba = rf2_model.predict_proba(X2_test)[:, 1]
best_thresh2, best_bal_acc2 = find_best_threshold(y2_test, y2_test_proba)

print(f"Model 2 - Best threshold: {best_thresh2:.3f}, Balanced Accuracy: {best_bal_acc2:.4f}")
print(f"OOB Score Model 2: {rf2_model.oob_score_:.4f}")


## MODEL COMPARISON

In [ ]:
# Detailed metrics evaluation
detailed_results = []
detailed_results.append(compute_detailed_metrics(y1_test, y1_test_proba, best_thresh1, "NoTracking"))
detailed_results.append(compute_detailed_metrics(y2_test, y2_test_proba, best_thresh2, "WithTracking"))

detailed_df = pd.DataFrame(detailed_results)
print("\nDetailed Metrics (Balanced Accuracy Optimal Thresholds):")
print("=" * 90)
detailed_df

In [ ]:
# Feature importance visualization
def plot_feature_importance(model, model_name, max_features=10):
    importances = model.feature_importances_
    feature_names = model.feature_names_in_
    
    indexes = np.argsort(importances)[::-1][:max_features]
    sorted_imp = importances[indexes]
    sorted_names = feature_names[indexes]
    
    plt.figure(figsize=(12, 6))
    bars = plt.bar(range(len(sorted_imp)), sorted_imp, align='center')
    plt.xticks(range(len(sorted_imp)), sorted_names, rotation=45, ha='right')
    plt.xlabel("Features")
    plt.ylabel("Importance")
    plt.title(f"Feature Importances - {model_name}")
    
    for i, bar in enumerate(bars):
        plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                 f'{sorted_imp[i]:.3f}', ha='center', va='bottom')
    
    plt.tight_layout()
    plt.show()

plot_feature_importance(rf1_model, "Random Forest No Tracking")

In [ ]:
plot_feature_importance(rf2_model, "Random Forest With Tracking")